In [1]:
from google.colab import files
uploaded = files.upload()

Saving prepared_ticket_training_data.csv to prepared_ticket_training_data.csv


In [2]:
!pip install transformers datasets scikit-learn accelerate -q

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [4]:
import pandas as pd

df = pd.read_csv("prepared_ticket_training_data.csv")

print(df.head())
print(df.shape)
print(df["Category"].value_counts())
print(df["Priority"].value_counts())

                                             Message  Category Priority
0  I need someone to look at an order that arrive...    Refund     High
1  This is urgent: a damaged order needed for an ...    Refund   Urgent
2  How should I handle connecting from a new offi...   Network      Low
3  We need this fixed now: a hacked account. This...  Security   Urgent
4  How should I handle business hours? I want to ...   General      Low
(11000, 3)
Category
Refund          1000
Network         1000
Security        1000
General         1000
Product         1000
Technical       1000
Subscription    1000
Account         1000
Outage          1000
Hardware        1000
Billing         1000
Name: count, dtype: int64
Priority
High      2750
Urgent    2750
Low       2750
Medium    2750
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

df["category_label"] = category_encoder.fit_transform(df["Category"])
df["priority_label"] = priority_encoder.fit_transform(df["Priority"])

category_labels = list(category_encoder.classes_)
priority_labels = list(priority_encoder.classes_)

print(category_labels)
print(priority_labels)

['Account', 'Billing', 'General', 'Hardware', 'Network', 'Outage', 'Product', 'Refund', 'Security', 'Subscription', 'Technical']
['High', 'Low', 'Medium', 'Urgent']


In [6]:
from sklearn.model_selection import train_test_split

df["stratify_label"] = df["Category"] + "_" + df["Priority"]

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["stratify_label"]
)

print(train_df.shape, val_df.shape)

(9350, 6) (1650, 6)


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [8]:
from torch.utils.data import Dataset

class TicketDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe["Message"].tolist()
        self.category_labels = dataframe["category_label"].tolist()
        self.priority_labels = dataframe["priority_label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "category_label": torch.tensor(self.category_labels[idx], dtype=torch.long),
            "priority_label": torch.tensor(self.priority_labels[idx], dtype=torch.long),
        }

train_dataset = TicketDataset(train_df, tokenizer)
val_dataset = TicketDataset(val_df, tokenizer)

In [9]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiTaskTicketModel(nn.Module):
    def __init__(self, model_name, num_categories, num_priorities):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.2)
        self.category_head = nn.Linear(hidden_size, num_categories)
        self.priority_head = nn.Linear(hidden_size, num_priorities)

    def forward(self, input_ids, attention_mask, category_label=None, priority_label=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        if hasattr(outputs, "last_hidden_state"):
            pooled = outputs.last_hidden_state[:, 0]
        else:
            pooled = outputs[0][:, 0]

        pooled = self.dropout(pooled)

        category_logits = self.category_head(pooled)
        priority_logits = self.priority_head(pooled)

        loss = None
        if category_label is not None and priority_label is not None:
            loss_fn = nn.CrossEntropyLoss()
            category_loss = loss_fn(category_logits, category_label)
            priority_loss = loss_fn(priority_logits, priority_label)
            loss = category_loss + priority_loss

        return {
            "loss": loss,
            "category_logits": category_logits,
            "priority_logits": priority_logits
        }

In [10]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskTicketModel(
    MODEL_NAME,
    num_categories=len(category_labels),
    num_priorities=len(priority_labels)
).to(device).float()

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

optimizer = AdamW(model.parameters(), lr=2e-5)

EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        category_label = batch["category_label"].to(device)
        priority_label = batch["priority_label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            category_label=category_label,
            priority_label=priority_label
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} loss: {total_loss / len(train_loader):.4f}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1: 100%|██████████| 585/585 [04:

...

Epoch 2 loss: 0.0197


Epoch 3: 100%|██████████| 585/585 [04:39<00:00,  2.10it/s]


Epoch 3 loss: 0.0085


Epoch 4: 100%|██████████| 585/585 [04:38<00:00,  2.10it/s]

Epoch 4 loss: 0.0049


In [11]:
from sklearn.metrics import classification_report

model.eval()

category_preds = []
category_true = []
priority_preds = []
priority_true = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        cat_pred = torch.argmax(outputs["category_logits"], dim=1).cpu().numpy()
        prio_pred = torch.argmax(outputs["priority_logits"], dim=1).cpu().numpy()

        category_preds.extend(cat_pred)
        priority_preds.extend(prio_pred)

        category_true.extend(batch["category_label"].numpy())
        priority_true.extend(batch["priority_label"].numpy())

print("Category Report")
print(classification_report(category_true, category_preds, target_names=category_labels))

print("Priority Report")
print(classification_report(priority_true, priority_preds, target_names=priority_labels))

Category Report
              precision    recall  f1-score   support

     Account       1.00      1.00      1.00       151
     Billing       1.00      1.00      1.00       150
     General       1.00      1.00      1.00       150
    Hardware       1.00      1.00      1.00       149
     Network       1.00      1.00      1.00       149
      Outage       1.00      1.00      1.00       152
     Product       1.00      1.00      1.00       148
      Refund       1.00      1.00      1.00       150
    Security       1.00      1.00      1.00       151
Subscription       1.00      1.00      1.00       150
   Technical       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

Priority Report
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       411
         Low       1.00      1.00      1.00   

In [12]:
import json
import os

SAVE_DIR = "ticket_multitask_model"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), f"{SAVE_DIR}/model_weights.pt")
tokenizer.save_pretrained(SAVE_DIR)

config = {
    "base_model": MODEL_NAME,
    "num_categories": len(category_labels),
    "num_priorities": len(priority_labels),
    "max_length": 128
}

label_mappings = {
    "categories": category_labels,
    "priorities": priority_labels
}

with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(f"{SAVE_DIR}/label_mappings.json", "w") as f:
    json.dump(label_mappings, f, indent=2)

print("Saved model to:", SAVE_DIR)

Saved model to: ticket_multitask_model


In [17]:
!zip -r ticket_multitask_model.zip ticket_multitask_model

from google.colab import files
files.download("ticket_multitask_model.zip")

  adding: ticket_multitask_model/ (stored 0%)
  adding: ticket_multitask_model/tokenizer.json (deflated 79%)
  adding: ticket_multitask_model/tokenizer_config.json (deflated 49%)
  adding: ticket_multitask_model/config.json (deflated 18%)
  adding: ticket_multitask_model/model_weights.pt (deflated 25%)
  adding: ticket_multitask_model/label_mappings.json (deflated 42%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from sklearn.metrics import accuracy_score, classification_report
import torch

def evaluate_model(model, val_loader, device, category_labels, priority_labels):
    model.eval()

    category_preds = []
    category_true = []

    priority_preds = []
    priority_true = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            cat_pred = torch.argmax(outputs["category_logits"], dim=1).cpu().numpy()
            prio_pred = torch.argmax(outputs["priority_logits"], dim=1).cpu().numpy()

            category_preds.extend(cat_pred)
            priority_preds.extend(prio_pred)

            category_true.extend(batch["category_label"].numpy())
            priority_true.extend(batch["priority_label"].numpy())

    category_accuracy = accuracy_score(category_true, category_preds) * 100
    priority_accuracy = accuracy_score(priority_true, priority_preds) * 100

    exact_match = sum(
        1
        for ct, cp, pt, pp in zip(
            category_true,
            category_preds,
            priority_true,
            priority_preds
        )
        if ct == cp and pt == pp
    )

    exact_match_accuracy = exact_match / len(category_true) * 100
    overall_accuracy = (category_accuracy + priority_accuracy) / 2

    print("=" * 60)
    print("MODEL PERFORMANCE")
    print("=" * 60)
    print(f"Category Accuracy:    {category_accuracy:.2f}%")
    print(f"Priority Accuracy:    {priority_accuracy:.2f}%")
    print(f"Overall Accuracy:     {overall_accuracy:.2f}%")
    print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")

    print("\n" + "=" * 60)
    print("CATEGORY REPORT")
    print("=" * 60)
    print(classification_report(
        category_true,
        category_preds,
        target_names=category_labels
    ))

    print("\n" + "=" * 60)
    print("PRIORITY REPORT")
    print("=" * 60)
    print(classification_report(
        priority_true,
        priority_preds,
        target_names=priority_labels
    ))

    return {
        "category_accuracy": category_accuracy,
        "priority_accuracy": priority_accuracy,
        "overall_accuracy": overall_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
    }

In [20]:
metrics = evaluate_model(
    model=model,
    val_loader=val_loader,
    device=device,
    category_labels=category_labels,
    priority_labels=priority_labels
)

MODEL PERFORMANCE
Category Accuracy:    100.00%
Priority Accuracy:    100.00%
Overall Accuracy:     100.00%
Exact Match Accuracy: 100.00%

CATEGORY REPORT
              precision    recall  f1-score   support

     Account       1.00      1.00      1.00       151
     Billing       1.00      1.00      1.00       150
     General       1.00      1.00      1.00       150
    Hardware       1.00      1.00      1.00       149
     Network       1.00      1.00      1.00       149
      Outage       1.00      1.00      1.00       152
     Product       1.00      1.00      1.00       148
      Refund       1.00      1.00      1.00       150
    Security       1.00      1.00      1.00       151
Subscription       1.00      1.00      1.00       150
   Technical       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650


PRIORITY REPORT
              p

In [21]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import DataLoader
import torch
import numpy as np

In [22]:
from sklearn.model_selection import train_test_split

df["stratify_label"] = df["Category"] + "_" + df["Priority"]

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["stratify_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["stratify_label"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (7700, 6)
Validation: (1650, 6)
Test: (1650, 6)


In [23]:
train_dataset = TicketDataset(train_df, tokenizer)
val_dataset = TicketDataset(val_df, tokenizer)
test_dataset = TicketDataset(test_df, tokenizer)

train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [24]:
def evaluate_split(model, data_loader, split_name):
    model.eval()

    total_loss = 0
    total_samples = 0

    category_true = []
    category_preds = []

    priority_true = []
    priority_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            category_label = batch["category_label"].to(device)
            priority_label = batch["priority_label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                category_label=category_label,
                priority_label=priority_label
            )

            loss = outputs["loss"]

            batch_size = input_ids.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            cat_pred = torch.argmax(outputs["category_logits"], dim=1)
            prio_pred = torch.argmax(outputs["priority_logits"], dim=1)

            category_true.extend(category_label.cpu().numpy())
            category_preds.extend(cat_pred.cpu().numpy())

            priority_true.extend(priority_label.cpu().numpy())
            priority_preds.extend(prio_pred.cpu().numpy())

    avg_loss = total_loss / total_samples

    category_acc = accuracy_score(category_true, category_preds) * 100
    priority_acc = accuracy_score(priority_true, priority_preds) * 100

    category_f1 = f1_score(category_true, category_preds, average="macro") * 100
    priority_f1 = f1_score(priority_true, priority_preds, average="macro") * 100

    exact_match = np.mean(
        [
            c_true == c_pred and p_true == p_pred
            for c_true, c_pred, p_true, p_pred in zip(
                category_true,
                category_preds,
                priority_true,
                priority_preds
            )
        ]
    ) * 100

    print("=" * 70)
    print(f"{split_name.upper()} SCORE")
    print("=" * 70)
    print(f"Loss:                 {avg_loss:.4f}")
    print(f"Category Accuracy:    {category_acc:.2f}%")
    print(f"Priority Accuracy:    {priority_acc:.2f}%")
    print(f"Category Macro F1:    {category_f1:.2f}%")
    print(f"Priority Macro F1:    {priority_f1:.2f}%")
    print(f"Exact Match Accuracy: {exact_match:.2f}%")

    return {
        "split": split_name,
        "loss": avg_loss,
        "category_accuracy": category_acc,
        "priority_accuracy": priority_acc,
        "category_macro_f1": category_f1,
        "priority_macro_f1": priority_f1,
        "exact_match_accuracy": exact_match,
        "category_true": category_true,
        "category_preds": category_preds,
        "priority_true": priority_true,
        "priority_preds": priority_preds,
    }

In [ ]:
train_scores = evaluate_split(model, train_eval_loader, "train")
val_scores = evaluate_split(model, val_loader, "validation")
test_scores = evaluate_split(model, test_loader, "test")

TRAIN SCORE
Loss:                 0.0007
Category Accuracy:    100.00%
Priority Accuracy:    100.00%
Category Macro F1:    100.00%
Priority Macro F1:    100.00%
Exact Match Accuracy: 100.00%
VALIDATION SCORE
Loss:                 0.0007
Category Accuracy:    100.00%
Priority Accuracy:    100.00%
Category Macro F1:    100.00%
Priority Macro F1:    100.00%
Exact Match Accuracy: 100.00%
TEST SCORE
Loss:                 0.0007
Category Accuracy:    100.00%
Priority Accuracy:    100.00%
Category Macro F1:    100.00%
Priority Macro F1:    100.00%
Exact Match Accuracy: 100.00%


In [ ]:
print("TEST CATEGORY REPORT")
print(classification_report(
    test_scores["category_true"],
    test_scores["category_preds"],
    target_names=category_labels
))

print("TEST PRIORITY REPORT")
print(classification_report(
    test_scores["priority_true"],
    test_scores["priority_preds"],
    target_names=priority_labels
))

In [14]:
import torch

def predict_message(message):
    model.eval()

    encoded = tokenizer(
        message,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        category_probs = torch.softmax(outputs["category_logits"], dim=1)[0]
        priority_probs = torch.softmax(outputs["priority_logits"], dim=1)[0]

        category_id = torch.argmax(category_probs).item()
        priority_id = torch.argmax(priority_probs).item()

    return {
        "message": message,
        "category": category_labels[category_id],
        "category_confidence": round(category_probs[category_id].item() * 100, 2),
        "priority": priority_labels[priority_id],
        "priority_confidence": round(priority_probs[priority_id].item() * 100, 2),
    }

In [16]:
result = predict_message("This is unacceptable! I have been waiting for over 48 hours for someone to fix my VPN connection. I can't do any remote work and my deadlines are ruining!")

print("Message:", result["message"])
print("Category:", result["category"])
print("Category Confidence:", result["category_confidence"], "%")
print("Priority:", result["priority"])
print("Priority Confidence:", result["priority_confidence"], "%")

Message: This is unacceptable! I have been waiting for over 48 hours for someone to fix my VPN connection. I can't do any remote work and my deadlines are ruining!
Category: Network
Category Confidence: 99.57 %
Priority: High
Priority Confidence: 99.94 %
